# Choosing k - number of clusters

This notebook wll provide experiemnts for internal validation metric to choose the number of clusters k.

In [1]:
# Main packages 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Clustering packages
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA


# Parallel processing packages
from joblib import Parallel, delayed

In [2]:
# Load data
df = pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [3]:
# Keep only human users
df = df.filter(pl.col("src_user").str.starts_with("U"))

We will start by fixing the aggregation level at 1 hour and the window size at 1 days. We will use 7 days of data to begin with, because of computational considerations, then will possibly extend to use large window sizes and more data. 

In [4]:
# Time conversions
SECONDS_IN_DAY = 60 * 60 * 24

### Tumbling Windows vs Sliding Windows

Justifications of using tumbling windows:
- Computationally cheaper
- Each window could be considered statistically independent
- Easier to evaluate, the metrics are easier to compare window T against window T + 1


From my thoughts: 
Suppose day 1 -7 and day 2 - 8. Then a point say at 'Tuesday 8am' in one window is then 'Monday 8am' in the next window, so not really comporable if clustering on [src_user, relative_bucket]


Need to find explicit references for this.


In [5]:
# FUNCTION - build features dataframe
def build_features(df,agg_hour_level):

    AGG_SECONDS = agg_hour_level * 60 * 60
    return (
        df.with_columns(
            bucket = pl.col('time') // AGG_SECONDS,
            theta = ((pl.col('time') % SECONDS_IN_DAY)/ SECONDS_IN_DAY) * 2 * np.pi,
            is_failure = (pl.col('outcome') == 'Fail').cast(pl.Int8),
        )
        .group_by(['src_user','bucket'])
        .agg(
            n_events = pl.len(),
            failure_ratio = pl.col('is_failure').mean(),
            n_distinct_dest = pl.col('dest_comp').n_unique(),
            n_distinct_src = pl.col('src_comp').n_unique(),
            c_bar = pl.col('theta').cos().mean(),
            s_bar = pl.col('theta').sin().mean(),
        )
        .with_columns(
             log_n_events=pl.col("n_events").log1p(),
             log_n_distinct_dest=pl.col("n_distinct_dest").log1p(),
             log_n_distinct_src=pl.col("n_distinct_src").log1p(),
        )
        .collect()
    )

In [6]:
# Create features dataframe
agg_hour_level = 1
features_df = build_features(df,agg_hour_level)

In [7]:
# Extract relevant features
feature_cols = [
    "log_n_events",
    "failure_ratio",
    "log_n_distinct_dest",
    "log_n_distinct_src",
    "c_bar",
    "s_bar",
]

In [8]:
# FUNCTION - process features for clustering 
def cluster_preprocess(features_df,feature_cols,agg_hour_level,week):

    buckets_per_week = (7 * 24) // agg_hour_level
    lb = (week - 1) * buckets_per_week
    ub = lb + buckets_per_week - 1

    features_week = features_df.filter(pl.col('bucket').is_between(lb,ub))

    X = features_week.select(feature_cols).to_numpy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return features_week, X_scaled

In [9]:
# Process features
features_week, X_scaled = cluster_preprocess(features_df,feature_cols,agg_hour_level,week = 1)

In [21]:
# FUNCTION - kmeans 
def fit_kmeans(k, Y, sample_size):
    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(Y)
    
    sil = silhouette_score(Y, labels, sample_size=sample_size, random_state=123)
    ch  = calinski_harabasz_score(Y, labels)   
    db  = davies_bouldin_score(Y, labels)    
    
    return k, sil,ch,db

In [37]:
# Run kmeans in parallel
sample_size = 100_000
results = Parallel(n_jobs=-1)(delayed(fit_kmeans)(k, X_scaled,sample_size) for k in range(2, 11))

In [38]:
metrics_df = pl.DataFrame(
    results,
    schema=["k", "silhouette", "calinski_harabasz", "davies_bouldin"],
    orient="row",
)

In [39]:
metrics_df

k,silhouette,calinski_harabasz,davies_bouldin
i64,f64,f64,f64
2,0.316891,344170.055828,1.377804
3,0.244172,272438.028659,1.449072
4,0.286159,333772.696818,1.202817
5,0.273652,321052.812638,1.096614
6,0.271992,318656.784218,1.026865
7,0.276459,312850.087809,1.121143
8,0.280641,302852.864328,1.05806
9,0.269698,297361.292723,1.066124
10,0.276627,292404.85498,1.054563


**How to select k**

Can use the majority rule, when dealing with multiple metrics, can take the optimal number of clusters by the majority rule.

Done in for example : Charrad M., Ghazzali N., Boiteau V., Niknafs A. (2014). NbClust: An R Package for Determining the Relevant Number of Clusters in a Data Set. Journal of Statistical Software, 61(6), 1–36.

Can also consider using more or different metrics, can come back to this potentially.


Thoughts:
- Use several metrics such as silhouette (sample), CH, DB
- Use a ranking system to pick k based on several metrics

Because I am only using a sample for the silhouette metric, wanted to use other metrics that are computationally cheaper in addition so the selection of k is more robust.

**initialisation**

How many initialisaitons and also consider the kmeans++ (i have left this to come back to)

## References

Arbelaitz et al. (2013) — "An extensive comparative study of cluster validity indices." Pattern Recognition 46:243–256. The benchmark reference for CVI selection.

- silhoutte performs well
- recommends using multiple validaiton criteria
- says they restrict number of clusters from k = 2 to x for computational reasons


Liu et al. (2010) — "Understanding of internal clustering validation measures." ICDM 2010, pp. 911–916. 

- metrics that use 'the elbow' are quite subjective

stability is used as a clustering validation or to pick the number of clusters 